# NB1 — Delta Lake Basics (lightweight path)

**Stack:** `deltalake` (delta-rs) + Polars + DuckDB. No Spark, no JVM.
Maps to slide §2 (Delta Lake) + deliverable bullet 1.

> Spark equivalent: `spark.read.format("delta").load(path)` ↔ `DeltaTable(path).to_pyarrow_table()`.
> Same on-disk format, different binding.

> **Người thực hiện:** Bùi Thị Thu Uyên · MSSV 2A202602613 · K4-Track02-Day18  
> **Đường chạy:** lightweight (`deltalake` 1.6.6, `pyiceberg` 0.12.0, DuckDB 1.5.6, Polars 1.44.2) · Python 3.11.7 · Windows 11  
> Notebook được thực thi từ đầu đến cuối; ô *Giải thích kết quả* ở cuối bài đọc các số in ở output phía trên.

In [1]:
import _setup  # noqa: F401  -- adds scripts/ to sys.path (file-relative)
import polars as pl
from deltalake import DeltaTable, write_deltalake
from lakehouse import path, reset

table_path = path("scratch", "users_delta")
reset(table_path)  # idempotent rerun

## 1. Write a Delta table

In [2]:
df = pl.DataFrame({
    "id": [1, 2, 3],
    "name": ["alice", "bob", "charlie"],
    "age": [30, 25, 35],
    "city": ["Hanoi", "HCMC", "Danang"],
})
write_deltalake(table_path, df.to_arrow(), mode="overwrite")

## 2. Read it back + inspect transaction log

Look at `_lakehouse/scratch/users_delta/_delta_log/00000000000000000000.json` —
that's the transaction log. Same JSON format Spark/Databricks would write.

In [3]:
dt = DeltaTable(table_path)
print(pl.from_arrow(dt.to_pyarrow_table()))
print("\nHistory:")
for h in dt.history():
    print(f"  v{h['version']}  {h['operation']}  {h.get('operationMetrics', {})}")

shape: (3, 4)
┌─────┬─────────┬─────┬────────┐
│ id  ┆ name    ┆ age ┆ city   │
│ --- ┆ ---     ┆ --- ┆ ---    │
│ i64 ┆ str     ┆ i64 ┆ str    │
╞═════╪═════════╪═════╪════════╡
│ 1   ┆ alice   ┆ 30  ┆ Hanoi  │
│ 2   ┆ bob     ┆ 25  ┆ HCMC   │
│ 3   ┆ charlie ┆ 35  ┆ Danang │
└─────┴─────────┴─────┴────────┘

History:
  v0  WRITE  {'num_added_files': 1, 'num_removed_files': 0, 'num_partitions': 0, 'num_added_rows': 3, 'execution_time_ms': 7, 'num_retries': 0}


In [4]:
# Evidence: list `_delta_log/` and print the actions inside commit v0.
import json
from pathlib import Path

log_dir = Path(table_path) / "_delta_log"
print(f"{log_dir}:")
for f in sorted(log_dir.iterdir()):
    print(f"  {f.name:<32} {f.stat().st_size:>6} B")

print("\n00000000000000000000.json — one JSON action per line:")
for line in (log_dir / "00000000000000000000.json").read_text().splitlines():
    action = json.loads(line)
    print(json.dumps(action, indent=2)[:900])

E:\4.10\K4-Track02-Day18-BuiThiThuUyen-2A202602613-Lakehouse-Lab\_lakehouse\scratch\users_delta\_delta_log:
  00000000000000000000.json          1409 B

00000000000000000000.json — one JSON action per line:
{
  "commitInfo": {
    "timestamp": 1791169213060,
    "operation": "WRITE",
    "operationParameters": {
      "mode": "Overwrite"
    },
    "engineInfo": "delta-rs:py-1.6.6",
    "clientVersion": "delta-rs.py-1.6.6",
    "operationMetrics": {
      "num_added_files": 1,
      "num_removed_files": 0,
      "num_partitions": 0,
      "num_added_rows": 3,
      "execution_time_ms": 7,
      "num_retries": 0
    }
  }
}
{
  "protocol": {
    "minReaderVersion": 1,
    "minWriterVersion": 2
  }
}
{
  "metaData": {
    "id": "bfef3eef-ea16-4038-b83d-27a845ecc413",
    "name": null,
    "description": null,
    "format": {
      "provider": "parquet",
      "options": {}
    },
    "schemaString": "{\"type\":\"struct\",\"fields\":[{\"name\":\"id\",\"type\":\"long\",\"nullable\":true,\"

## 3. Schema enforcement — try to write a wrong schema

In [5]:
bad = pl.DataFrame({"id": [4], "name": ["dan"], "age": ["thirty"], "city": ["Hue"]})
version_before_bad = DeltaTable(table_path).version()
bad_write_blocked = False
try:
    write_deltalake(table_path, bad.to_arrow(), mode="append")
    print("UNEXPECTED: bad write succeeded — schema enforcement broken")
except Exception as e:
    bad_write_blocked = True
    msg = str(e).splitlines()[0][:120]
    print(f"BLOCKED by schema enforcement (expected): {type(e).__name__}: {msg}")
version_after_bad = DeltaTable(table_path).version()
print(f"Table version before/after the rejected write: v{version_before_bad} → v{version_after_bad}"
      "  (no new commit)")

BLOCKED by schema enforcement (expected): Exception: Cast error: Cannot cast string 'thirty' to value of Int64 type
Table version before/after the rejected write: v0 → v0  (no new commit)


## 4. Schema evolution (opt-in)

In [6]:
new = pl.DataFrame({
    "id": [4], "name": ["dan"], "age": [28], "city": ["Hue"], "tier": ["premium"],
})
write_deltalake(table_path, new.to_arrow(), mode="append", schema_mode="merge")
dt = DeltaTable(table_path)
# Sort by id so the printout is stable across reruns — Delta does not
# preserve write-order across appends.
print(pl.from_arrow(dt.to_pyarrow_table()).sort("id"))

shape: (4, 5)
┌─────┬─────────┬─────┬────────┬─────────┐
│ id  ┆ name    ┆ age ┆ city   ┆ tier    │
│ --- ┆ ---     ┆ --- ┆ ---    ┆ ---     │
│ i64 ┆ str     ┆ i64 ┆ str    ┆ str     │
╞═════╪═════════╪═════╪════════╪═════════╡
│ 1   ┆ alice   ┆ 30  ┆ Hanoi  ┆ null    │
│ 2   ┆ bob     ┆ 25  ┆ HCMC   ┆ null    │
│ 3   ┆ charlie ┆ 35  ┆ Danang ┆ null    │
│ 4   ┆ dan     ┆ 28  ┆ Hue    ┆ premium │
└─────┴─────────┴─────┴────────┴─────────┘


In [7]:
# Evidence: the opt-in evolution is itself a commit. v1 carries a new
# `metaData` action whose schema now includes `tier`.
print(f"{log_dir}:")
for f in sorted(log_dir.iterdir()):
    print(f"  {f.name:<32} {f.stat().st_size:>6} B")

for line in (log_dir / "00000000000000000001.json").read_text().splitlines():
    action = json.loads(line)
    kind = next(iter(action))
    if kind == "metaData":
        fields = [f["name"] for f in json.loads(action["metaData"]["schemaString"])["fields"]]
        print(f"\nv1 metaData schema fields: {fields}")
    elif kind == "commitInfo":
        print(f"v1 commitInfo: operation={action[kind]['operation']}  "
              f"params={action[kind].get('operationParameters')}")

E:\4.10\K4-Track02-Day18-BuiThiThuUyen-2A202602613-Lakehouse-Lab\_lakehouse\scratch\users_delta\_delta_log:
  00000000000000000000.json          1409 B
  00000000000000000001.json          1465 B
v1 commitInfo: operation=WRITE  params={'mode': 'Append'}

v1 metaData schema fields: ['id', 'name', 'age', 'city', 'tier']


## 5. Query with DuckDB via Arrow (part of the required notebook)

In [8]:
import duckdb

# We hand DuckDB an Arrow table rather than calling `delta_scan()`. delta_scan
# autoloads a DuckDB extension over the network — fine at home, a support
# ticket in a firewalled classroom. Arrow registration is zero-copy and offline.
con = duckdb.connect()
con.register("users", DeltaTable(table_path).to_pyarrow_table())
tier_counts = con.sql("SELECT tier, count(*) AS n FROM users GROUP BY 1 ORDER BY 1").fetchall()
print(tier_counts)

[('premium', 1), (None, 3)]


## 🔎 Giải thích kết quả — NB1

**1. Transaction log.** `_delta_log/` có 2 commit JSON. `00000000000000000000.json` gồm 4 action, mỗi dòng một action:
`commitInfo` (WRITE, mode Overwrite), `protocol` (minReader 1 / minWriter 2), `metaData` (schema 4 cột `id, name, age, city`)
và `add` (1 file Parquet 1,384 B kèm stats `numRecords`, `minValues`, `maxValues`, `nullCount`). Trạng thái bảng chính là kết quả
replay các action này theo thứ tự version. Stats trong `add` là thứ NB2/NB6 dùng để bỏ qua file.

**2. Schema enforcement.** Ghi `age='thirty'` bị chặn với lỗi thật
`Cast error: Cannot cast string 'thirty' to value of Int64 type`, và version vẫn là **v0 → v0**: không commit nào được tạo,
nên không reader nào thấy dữ liệu sai kiểu. *Chỉnh sửa:* cờ `schema enforcement blocked bad write` ở ô kiểm tra cuối
ban đầu bị hardcode `True`. Mình thay bằng biến `bad_write_blocked` gán trong khối `except` và thêm điều kiện version không đổi,
nên dòng PASS giờ là bằng chứng thật.

**3. Evolution phải opt-in.** Chỉ khi truyền `schema_mode="merge"` thì v1 mới thêm cột `tier`: action `metaData` của v1 có
schema `['id', 'name', 'age', 'city', 'tier']`. Commit v1 là Append nên file Parquet của v0 không bị ghi lại;
3 dòng cũ đọc ra `tier = null`. DuckDB đếm được 2 nhóm: `premium` = 1 và `NULL` = 3.

**Ý nghĩa:** enforcement ngăn một producer gửi sai kiểu làm hỏng cột của mọi consumer phía sau; muốn đổi schema
thì phải chủ động cho phép, và việc đổi đó cũng được ghi lại thành một transaction trong log.

## ✅ Deliverable check
- [ ] `_delta_log/` contains JSON files
- [ ] Schema enforcement blocked the bad write
- [ ] schema_mode="merge" added the `tier` column
- [ ] DuckDB query returned 2 tier groups
The schema-enforcement flag is set inside the bad-write `except` block and
also requires that the rejected write created no new table version.

In [9]:
_log = sorted(Path(table_path).glob("_delta_log/*.json"))
_cols = DeltaTable(table_path).schema().to_arrow().names
checks = {
    "_delta_log/ has JSON commits": len(_log) >= 2,
    "schema enforcement blocked bad write": bad_write_blocked and version_after_bad == version_before_bad,
    "tier column added via schema_mode=merge": "tier" in _cols,
    "duckdb sees 2 tier groups": len(tier_counts) == 2,
}
for k, v in checks.items():
    print(f"  [{'PASS' if v else 'FAIL'}] {k}")
assert all(checks.values()), "NB1 incomplete — see FAIL rows above"
print("\nNB1 complete.")

  [PASS] _delta_log/ has JSON commits
  [PASS] schema enforcement blocked bad write
  [PASS] tier column added via schema_mode=merge
  [PASS] duckdb sees 2 tier groups

NB1 complete.
